# DataTrust — Auditoría de calidad: tabla `calendar`

## Contexto

La tabla `calendar` contiene 5.623.190 filas: un registro por cada combinación de alojamiento y día, con la disponibilidad (`available`) y la estancia mínima/máxima permitida ese día (`minimum_nights`, `maximum_nights`). Cubre 365 días de calendario por alojamiento.

## Problema

Esta tabla es relevante para cualquier cálculo de ocupación o ingresos basado en disponibilidad. `listings` ya incluye una variable de negocio relacionada, `estimated_revenue_l365d`, pero este notebook no verifica cómo se calculó esa variable internamente ni afirma que dependa directamente de `calendar`. Lo que sí es cierto es que cualquier análisis propio que se apoye en esta tabla (por ejemplo, calcular ocupación real por alojamiento) hereda los problemas que tenga `calendar`: registros huérfanos, estructura inconsistente, o valores incoherentes entre `minimum_nights` y `maximum_nights`.

## Objetivo de este notebook

Auditar `calendar` en cuatro dimensiones: completitud, duplicados y estructura (filas por alojamiento), integridad referencial (con SQL, contra `listings`) y coherencia (`minimum_nights` frente a `maximum_nights`). No se corrige nada todavía.

In [1]:
import pandas as pd
import sqlite3

## Nota metodológica: por qué este notebook usa SQL de principio a fin

`calendar` tiene 5.623.190 filas, unas 5 veces más que `reviews` y 367 veces más que `listings`. Cargar el CSV completo en pandas es posible, pero las consultas de agregación (conteos, `GROUP BY`, `JOIN`) son más rápidas y más claras directamente en SQL sobre la base de datos ya construida (`database/airbnb.db`). Por eso, a diferencia de los notebooks 1 y 2, aquí no se carga el CSV de `calendar` en memoria con pandas: todas las comprobaciones se hacen con `sqlite3` y `pd.read_sql`.

In [2]:
con = sqlite3.connect("../database/airbnb.db")

total_filas = pd.read_sql("SELECT COUNT(*) AS total FROM calendar", con)
print(total_filas)

     total
0  5623190


## 1. Completitud

**Criterio (mismo umbral que en los notebooks anteriores, para mantener consistencia)**: 100% de nulos → caso crítico; ≥10% y <100% → problema de completitud a documentar (umbral operativo definido para este proyecto, no un estándar externo); <10% → no se reporta como hallazgo relevante.

In [3]:
query_nulos = """
SELECT
    SUM(CASE WHEN listing_id IS NULL THEN 1 ELSE 0 END) AS listing_id,
    SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS date,
    SUM(CASE WHEN available IS NULL THEN 1 ELSE 0 END) AS available,
    SUM(CASE WHEN minimum_nights IS NULL THEN 1 ELSE 0 END) AS minimum_nights,
    SUM(CASE WHEN maximum_nights IS NULL THEN 1 ELSE 0 END) AS maximum_nights
FROM calendar
"""
print(pd.read_sql(query_nulos, con))

   listing_id  date  available  minimum_nights  maximum_nights
0           0     0          0               0               0


**Interpretación**: no se detectan valores nulos en ninguna de las cinco columnas auditadas (`listing_id`, `date`, `available`, `minimum_nights`, `maximum_nights`). La completitud es del 100% en todas ellas, por lo que no se reporta ningún hallazgo de completitud.

## 2. Duplicados y estructura

**Criterio**: no debería haber filas duplicadas para la combinación (`listing_id`, `date`), cada alojamiento debería tener como mucho un registro por día. Además, dado que el snapshot cubre un periodo fijo, cada alojamiento debería tener el mismo número de filas (un año de calendario). También se comprueba que `available` solo contenga los valores esperados.

In [4]:
dup_query = """
SELECT COUNT(*) AS combinaciones_duplicadas FROM (
    SELECT listing_id, date, COUNT(*) AS n
    FROM calendar
    GROUP BY listing_id, date
    HAVING n > 1
)
"""
print(pd.read_sql(dup_query, con))

filas_por_listing = pd.read_sql("""
SELECT MIN(n) AS minimo, MAX(n) AS maximo, COUNT(DISTINCT n) AS valores_distintos
FROM (SELECT listing_id, COUNT(*) AS n FROM calendar GROUP BY listing_id)
""", con)
print()
print(filas_por_listing)

valores_available = pd.read_sql("SELECT available, COUNT(*) AS n FROM calendar GROUP BY available", con)
print()
print(valores_available)

   combinaciones_duplicadas
0                         0



   minimo  maximo  valores_distintos
0     365     365                  1



  available        n
0         f  2342152
1         t  3281038


**Interpretación**: no hay combinaciones (`listing_id`, `date`) duplicadas. Además, todos los alojamientos tienen exactamente 365 registros (mínimo = máximo = 365 y un único valor distinto de conteo), por lo que la estructura esperada de un registro por alojamiento y día se cumple sin huecos ni registros adicionales. La columna `available` solo contiene los dos valores esperados ('t'/'f'), sin categorías inválidas. Las tres verificaciones dan resultado satisfactorio, no se reportan como hallazgos.

## 3. Integridad referencial

**Criterio**: todo `listing_id` de `calendar` debería existir en `listings`. Se comprueba con el mismo patrón `LEFT JOIN` usado en el notebook de `reviews`, la forma natural de detectar registros huérfanos.

In [5]:
huerfanos_query = """
SELECT COUNT(*) AS filas_huerfanas, COUNT(DISTINCT c.listing_id) AS listings_huerfanos
FROM calendar c
LEFT JOIN listings l ON c.listing_id = l.id
WHERE l.id IS NULL
"""
print(pd.read_sql(huerfanos_query, con))

   filas_huerfanas  listings_huerfanos
0            41245                 113


**Interpretación**: 41.245 filas de `calendar` (0,73% del total) pertenecen a 113 `listing_id` que no existen en el snapshot actual de `listings`. Como en `reviews`, una posible explicación es que esos anuncios hayan sido dados de baja después de generarse el calendario, aunque el dataset por sí solo no permite confirmar la causa.

Nota de consistencia: dado que cada alojamiento tiene exactamente 365 filas (Sección 2), 113 listings × 365 = 41.245, exactamente la cifra observada. Esto confirma que cada uno de esos 113 listings tiene 365 registros en `calendar`, no es un problema de carga parcial de datos.

Es un **riesgo de integridad referencial**: estas filas no pueden vincularse con los atributos correspondientes de `listings` (precio, distrito, tipo de anfitrión, etc.). Por tanto, cualquier análisis de `calendar` que necesite esos atributos debe tratar previamente estos registros huérfanos.

## 4. Coherencia

**Criterio**: `minimum_nights` no debería ser mayor que `maximum_nights` en la misma fila, sería una condición contradictoria (exigir una estancia mínima más larga que la máxima permitida). Es un **valor imposible** según el marco de tres niveles definido en el notebook de `listings` (extremo válido / sospechoso / imposible): no depende de un límite de negocio externo, es una contradicción lógica interna del propio dato.

In [6]:
incoherencia_query = """
SELECT COUNT(*) AS filas_incoherentes, COUNT(DISTINCT listing_id) AS listings_afectados
FROM calendar
WHERE minimum_nights > maximum_nights
"""
print(pd.read_sql(incoherencia_query, con))

ejemplos = pd.read_sql("""
SELECT listing_id, date, minimum_nights, maximum_nights
FROM calendar
WHERE minimum_nights > maximum_nights
LIMIT 5
""", con)
print()
print(ejemplos)

top_maximum_nights = pd.read_sql("""
SELECT maximum_nights, COUNT(*) AS n
FROM calendar
GROUP BY maximum_nights
ORDER BY n DESC
LIMIT 5
""", con)
print()
print(top_maximum_nights)

   filas_incoherentes  listings_afectados
0                 746                   6

   listing_id        date  minimum_nights  maximum_nights
0    13229284  2026-07-02              31              30
1    13229284  2026-07-03              31              30
2    13229284  2026-07-04              31              30
3    13229284  2026-07-05              31              30
4    13229284  2026-07-06              31              30



   maximum_nights        n
0             365  1562582
1            1125  1478636
2             999   385452
3             330   296388
4             125   214718


**Interpretación**: 746 filas (0,013% del total) tienen `minimum_nights` mayor que `maximum_nights`, una contradicción lógica. Estas filas se concentran en solo 6 alojamientos (una media de unas 124 filas cada uno), no están dispersas al azar entre muchos anuncios distintos: sugiere un problema de configuración sistemático asociado a esos 6 anuncios concretos, no ruido aleatorio disperso por todo el dataset.

Por separado, se observa que los valores más frecuentes de `maximum_nights` son 365 (1.562.582 filas), 1.125 (1.478.636 filas) y 999 (385.452 filas), tres cifras redondas que se repiten en una proporción muy alta de filas. Su alta frecuencia y su apariencia de valores límite (más que duraciones elegidas caso por caso) son compatibles con que funcionen como valores por defecto o límites administrativos amplios de la plataforma, en vez de una decisión individual de cada anfitrión, aunque el dataset por sí solo no permite confirmar su significado exacto. Se documenta como observación, no como hallazgo con umbral objetivo (mismo criterio aplicado a `price` en el notebook de `listings`: sin fuente que confirme el límite, no se clasifica como problema).

## Resumen de hallazgos — tabla `calendar`

| # | Hallazgo | Dimensión | Cifra |
|---|---|---|---|
| C1 | Filas huérfanas (`LEFT JOIN` contra `listings`) | Integridad referencial | 41.245 (0,73%), 113 listings |
| C2 | `minimum_nights` > `maximum_nights`, valor imposible | Coherencia | 746 filas, 6 listings |

**Verificaciones con resultado satisfactorio** (se comprobaron y no se detectaron problemas de calidad, por lo que no se incluyen en la tabla de hallazgos):

- Nulos en las 5 columnas: 0.
- Estructura: no existen combinaciones (`listing_id`, `date`) duplicadas y todos los alojamientos tienen exactamente 365 filas, por lo que se cumple la estructura esperada de un registro por alojamiento y día, sin huecos ni registros adicionales.
- `available`: solo contiene los dos valores esperados ('t'/'f'), sin categorías inválidas.

Estos 2 hallazgos pasan a la tabla de recomendaciones de gobernanza, junto con los de `listings` y `reviews`.